In [2]:
# First let's do an import. If you get an Import Error, double check that your Kernel is correct..

from dotenv import load_dotenv


In [3]:
# Next it's time to load the API keys into environment variables
# If this returns false, see the next cell!

load_dotenv(override=True)

True

In [4]:
# Check required credentials/endpoints for the 3 providers in this notebook

import os

groq_api_key = os.getenv("GROQ_API_KEY")
vllm_base_url = os.getenv("VLLM_BASE_URL", "http://192.168.0.80:31080/v1")
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://192.168.0.183:31435")

if groq_api_key:
    print(f"GROQ_API_KEY exists and begins {groq_api_key[:6]}")
else:
    print("GROQ_API_KEY not set - please add it to .env")

print(f"vLLM base URL: {vllm_base_url}")
print(f"Ollama base URL: {ollama_base_url}")

GROQ_API_KEY exists and begins gsk_pu
vLLM base URL: http://localhost:8000/v1
Ollama base URL: http://192.168.0.183:31435


In [5]:
# Import the SDKs used to call Groq and vLLM (OpenAI-compatible) and Ollama (native API)

from openai import OpenAI
import requests

In [ ]:
# Configure clients for the 3 APIs we are using: Groq, vLLM, and Ollama

groq = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
 )

vllm = OpenAI(
    api_key=os.getenv("VLLM_API_KEY", "not-needed"),
    base_url=os.getenv("VLLM_BASE_URL", "http://10.43.55.74:8000/v1")
 )

# OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://192.168.0.183:31435")
OLLAMA_BASE_URL = "http://192.168.0.183:31435"

MODELS = {
    "groq": os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
    "vllm": os.getenv("VLLM_MODEL", "tinyllama-chat"),
    "ollama": os.getenv("OLLAMA_MODEL", "qwen2.5:7b"),
}

In [7]:
# Create a list of messages in the familiar OpenAI format

messages = [{"role": "user", "content": "What is 2+2?"}]

In [17]:
client = OpenAI(api_key="", base_url=f"{OLLAMA_BASE_URL}/v1")
resp = client.chat.completions.create(
    model="qwen2.5:7b",
    messages=messages,
    # max_tokens=5000
)
print(resp.choices[0].message.content)

Let's denote the cost of the ball as \( x \) dollars.

Given that the bat costs $1.00 more than the ball, we can express the cost of the bat as \( x + 1 \) dollars.

The total cost of the bat and the ball together is given as $1.10. Therefore, we can write the equation:
\[ x + (x + 1) = 1.10 \]

Simplifying this equation gives us:
\[ 2x + 1 = 1.10 \]
Subtracting 1 from both sides results in:
\[ 2x = 0.10 \]
Dividing both sides by 2, we get:
\[ x = 0.05 \]

So, the ball costs $0.05 (or 5 cents).

To verify:
- The ball costs $0.05.
- The bat costs $1.00 more than the ball, so it costs $1.05.
- Together, they cost \( 0.05 + 1.05 = 1.10 \), which matches the given total.

Therefore, the ball costs $0.05.


In [8]:
# Call all 3 providers with the same prompt

def call_provider(provider_name, messages, max_tokens=256):
    if provider_name == "groq":
        response = groq.chat.completions.create(
            model=MODELS["groq"],
            messages=messages,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    if provider_name == "vllm":
        response = vllm.chat.completions.create(
            model=MODELS["vllm"],
            messages=messages,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    if provider_name == "ollama":
        prompt = messages[-1]["content"]
        payload = {
            "model": MODELS["ollama"],
            "prompt": prompt,
            "stream": False,
        }
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=120,
        )
        response.raise_for_status()
        return response.json().get("response", "")

    raise ValueError(f"Unknown provider: {provider_name}")

provider_answers = {}
for provider_name in ["groq", "vllm", "ollama"]:
    try:
        provider_answers[provider_name] = call_provider(provider_name, messages)
        print(f"{provider_name.upper()} response:\n{provider_answers[provider_name]}\n")
    except Exception as e:
        print(f"{provider_name.upper()} failed: {e}\n")

GROQ response:
2 + 2 = 4.

VLLM response:
2 + 2 = 4.

OLLAMA response:
2 + 2 equals 4.



In [9]:
# And now - let's ask for a question:

question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]


In [10]:
# Ask each provider for a challenging question and select one

provider_questions = {}
for provider_name in ["groq", "vllm", "ollama"]:
    try:
        provider_questions[provider_name] = call_provider(provider_name, messages, max_tokens=200)
        print(f"Question from {provider_name.upper()}:\n{provider_questions[provider_name]}\n")
    except Exception as e:
        print(f"{provider_name.upper()} failed while generating question: {e}\n")

if not provider_questions:
    raise RuntimeError("No provider returned a question.")

question = provider_questions.get("groq") or next(iter(provider_questions.values()))
print(f"Selected question:\n{question}")

Question from GROQ:
A bat and a ball together cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?

Question from VLLM:
Question: The first thing your future colleague will ask you is "Why do you want to choose our company?" Show your answer using bold, active, and aggressive language to impress their attention. Don't hold back!

Question from OLLAMA:
What is the smallest positive integer that is divisible by both 18 and 30 but not by 45?

Selected question:
A bat and a ball together cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost?


In [11]:
# form a new messages list
messages = [{"role": "user", "content": question}]


In [12]:
# Ask all providers the same selected question

answers = {}
for provider_name in ["groq", "vllm", "ollama"]:
    try:
        answers[provider_name] = call_provider(provider_name, messages)
        print(f"{provider_name.upper()} answer:\n{answers[provider_name]}\n")
    except Exception as e:
        print(f"{provider_name.upper()} failed while answering: {e}\n")

if not answers:
    raise RuntimeError("No provider returned an answer.")

# Keep compatibility with the next display cell
answer = answers.get("groq") or next(iter(answers.values()))
print("Chosen answer for legacy variable 'answer':")
print(answer)

GROQ answer:
To find the cost of the ball, let's denote the cost of the ball as x.

The bat costs $1.00 more than the ball, so the cost of the bat is x + $1.00.

The total cost of the bat and the ball is $1.10, so we can write the equation:

x + (x + $1.00) = $1.10

Combine like terms:
2x + $1.00 = $1.10

Subtract $1.00 from both sides:
2x = $0.10

Divide both sides by 2:
x = $0.05

The ball costs $0.05.

VLLM answer:
The ball cost $1.10 more than the bat, and the total cost of the bat and ball together is $1.10 + $1.00 = $1.10. Therefore, the cost of the ball is $1.10.

OLLAMA answer:
Let's break this down step-by-step:

1. Let the cost of the ball be \( B \) dollars.
2. According to the problem, the bat costs $1.00 more than the ball, so the cost of the bat is \( B + 1.00 \) dollars.

The total cost of both the bat and the ball together is given as $1.10. Therefore, we can write the equation:

\[ B + (B + 1.00) = 1.10 \]

Simplifying this equation:

\[ 2B + 1.00 = 1.10 \]

Subtract $

In [13]:
from IPython.display import Markdown, display

for provider_name, provider_answer in answers.items():
    display(Markdown(f"### {provider_name.upper()}"))
    display(Markdown(provider_answer))

### GROQ

To find the cost of the ball, let's denote the cost of the ball as x.

The bat costs $1.00 more than the ball, so the cost of the bat is x + $1.00.

The total cost of the bat and the ball is $1.10, so we can write the equation:

x + (x + $1.00) = $1.10

Combine like terms:
2x + $1.00 = $1.10

Subtract $1.00 from both sides:
2x = $0.10

Divide both sides by 2:
x = $0.05

The ball costs $0.05.

### VLLM

The ball cost $1.10 more than the bat, and the total cost of the bat and ball together is $1.10 + $1.00 = $1.10. Therefore, the cost of the ball is $1.10.

### OLLAMA

Let's break this down step-by-step:

1. Let the cost of the ball be \( B \) dollars.
2. According to the problem, the bat costs $1.00 more than the ball, so the cost of the bat is \( B + 1.00 \) dollars.

The total cost of both the bat and the ball together is given as $1.10. Therefore, we can write the equation:

\[ B + (B + 1.00) = 1.10 \]

Simplifying this equation:

\[ 2B + 1.00 = 1.10 \]

Subtract $1.00 from both sides to isolate the term with \( B \):

\[ 2B = 0.10 \]

Now, divide by 2 to solve for \( B \):

\[ B = \frac{0.10}{2} = 0.05 \]

So, the ball costs $0.05 (or 5 cents).

To verify:

- The ball costs $0.05.
- The bat costs $1.05 (which is $1.00 more than the ball).
- Together they cost \( 0.05 + 1.05 = 1.10 \), which matches the given total.

Therefore, the ball costs $0.05.

In [ ]:
# First create the messages:

messages = [{"role": "user", "content": "Something here"}]

# Then make the first call:

response =

# Then read the business idea:

business_idea = response.

# And repeat! In the next message, include the business idea within the message